# Undetermined Coefficients: Guessing Right, and When the Guess Has to Change

**Time:** about 15 to 20 minutes.
**You need:** nothing but this page. New to notebooks? Do `00-start-here.ipynb` first.

This activity goes with Trench section 5.4. The equations all look like

$$ay''+by'+cy=e^{\alpha x}G(x),$$

with constants $a,b,c,\alpha$ and a polynomial $G$. The method of undetermined coefficients says:
guess that $y_p$ looks like the right side, plug it in, and solve for the unknown numbers in the
guess. Usually that works. Sometimes the obvious guess is impossible to fix no matter what numbers
you pick, and the rule is to multiply it by $x$, or by $x^2$.

> **The main idea:** plug in $y=u\,e^{\alpha x}$ and every $e^{\alpha x}$ cancels, leaving
> $$a\,u''+p'(\alpha)\,u'+p(\alpha)\,u=G(x),$$
> where $p(r)=ar^2+br+c$ is the characteristic polynomial. Whether you need the extra $x$ depends
> only on whether $p(\alpha)$ and $p'(\alpha)$ are zero.

You will see why that is true by tuning guesses by hand, watching a guess fail no matter what you
do, and then reading the answer straight off the graph of $p$.

Each section asks you to predict before you run anything. Commit to an answer first, on paper or
out loud. Being wrong and seeing why is the point.

## Run this first

The two cells below set up the drawing tools. You never need to read or change them. Run them
once each (**Shift + Enter**) and give them a moment on the first run.

In [ ]:
%pip install -q ipywidgets sympy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from numpy.polynomial import Polynomial
from IPython.display import display, Markdown

X = sp.symbols('x')


def fmt_poly(P):
    'A numpy Polynomial as readable text, highest power first.'
    terms = []
    for k in range(len(P.coef) - 1, -1, -1):
        c = P.coef[k]
        if abs(c) < 1e-9:
            continue
        num = '' if (k > 0 and abs(abs(c) - 1) < 1e-9) else f'{abs(c):g}'
        power = '' if k == 0 else ('x' if k == 1 else f'x^{k}')
        terms.append(('-' if c < 0 else '+', num + power))
    if not terms:
        return '0'
    s = ('-' if terms[0][0] == '-' else '') + terms[0][1]
    for sign, t in terms[1:]:
        s += f' {sign} {t}'
    return s


def exp_label(alpha):
    'e^(alpha x) as readable text.'
    if alpha == 0:
        return ''
    if alpha == 1:
        return 'e^x'
    if alpha == -1:
        return 'e^(-x)'
    return f'e^({alpha:g}x)'


def leftover(a, b, c, alpha, u, G):
    'For y = u e^(alpha x): the leftover L[y] - e^(alpha x) G, divided by e^(alpha x).'
    p_alpha = a*alpha**2 + b*alpha + c
    dp_alpha = 2*a*alpha + b
    return a*u.deriv(2) + dp_alpha*u.deriv(1) + p_alpha*u - G


def leftover_plot(a, b, c, alpha, u, G, guess, x_range, lims):
    'Left: the leftover L[y_p] - f. Right: the same leftover with e^(alpha x) divided out.'
    R = leftover(a, b, c, alpha, u, G)
    x = np.linspace(*x_range, 400)
    matched = np.all(np.abs(R.coef) < 1e-9)
    color = 'C2' if matched else 'C3'

    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 4.2))
    for ax, ys, lim, name in ((axl, np.exp(alpha*x)*R(x), lims[0], "leftover  L[y_p] - f(x)"),
                              (axr, R(x), lims[1], f'leftover divided by {exp_label(alpha)}')):
        ax.axhline(0, color='0.4', linewidth=1)
        ax.plot(x, ys, color=color, linewidth=3 if matched else 2.2)
        ax.set_xlim(x_range)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel('x')
        ax.set_title(name)
        ax.grid(alpha=0.25)

    if matched:
        status = 'MATCH: the leftover is zero for every x, so this y_p solves the equation'
    else:
        status = f'leftover / {exp_label(alpha)} = {fmt_poly(R)}     (not zero yet)'
    fig.suptitle(f'guess:  y_p = {guess}\n{status}', fontsize=11)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def particular(a, b, c, alpha, G):
    "Solve the reduced equation for u. Returns (u, extra powers of x, p(alpha), p'(alpha))."
    a, b, c, alpha = (sp.nsimplify(v) for v in (a, b, c, alpha))
    G = sp.expand(G)
    p_alpha = a*alpha**2 + b*alpha + c
    dp_alpha = 2*a*alpha + b
    s = 0 if p_alpha != 0 else (1 if dp_alpha != 0 else 2)
    k = sp.degree(G, X) if G != 0 else 0
    cs = sp.symbols(f'k0:{k + 1}')
    u = X**s * sum(cs[i]*X**i for i in range(k + 1))
    eq = sp.expand(a*u.diff(X, 2) + dp_alpha*u.diff(X) + p_alpha*u - G)
    sol = sp.solve(sp.Poly(eq, X).all_coeffs(), cs, dict=True)[0]
    u = sp.expand(u.subs(sol).subs({ci: 0 for ci in cs}))
    return u, s, p_alpha, dp_alpha


def guess_form(alpha, k, s):
    'The guess with letters: x^s (A + Bx + ...) e^(alpha x), in LaTeX.'
    letters = sp.symbols('A B C D E F')
    body = sum(letters[i]*X**i for i in range(k + 1))
    return sp.latex(X**s * body * sp.exp(sp.nsimplify(alpha)*X))


def lin_latex(coefs, names):
    'A combination like 2u\'\' - u\' + 3u in LaTeX, highest derivative first, zero terms dropped.'
    out = ''
    for coef, name in zip(coefs, names):
        coef = sp.nsimplify(coef)
        if coef == 0:
            continue
        mag = '' if abs(coef) == 1 else sp.latex(abs(coef))
        sign = '-' if coef < 0 else '+'
        out += (('-' if sign == '-' else '') if not out else f' {sign} ') + mag + name
    return out


def reduced_latex(a, dp_alpha, p_alpha, G):
    "The equation a u'' + p'(alpha) u' + p(alpha) u = G, in LaTeX."
    return lin_latex((a, dp_alpha, p_alpha), ("u''", "u'", 'u')) + ' = ' + sp.latex(G)


CASES = {0: r'$p(\alpha)\neq 0$, so the $u$ term survives and $u$ has the same degree as $G$. '
            r'No extra $x$.',
         1: r"$p(\alpha)=0$, so the $u$ term is gone. The lowest surviving term is $u'$, which "
            r'lowers the degree by one, so $u$ needs one more power of $x$ than $G$.',
         2: r"$p(\alpha)=0$ and $p'(\alpha)=0$, so only $u''$ is left. It lowers the degree by "
            r'two, so $u$ needs two more powers of $x$ than $G$.'}


def homogeneous(a, b, c):
    "Two independent solutions of a y'' + b y' + c y = 0, as sympy expressions."
    a, b, c = (sp.nsimplify(v) for v in (a, b, c))
    disc = b**2 - 4*a*c
    if disc > 0:
        r1, r2 = (-b - sp.sqrt(disc))/(2*a), (-b + sp.sqrt(disc))/(2*a)
        return sp.exp(r1*X), sp.exp(r2*X)
    if disc == 0:
        r = -b/(2*a)
        return sp.exp(r*X), X*sp.exp(r*X)
    lam, mu = -b/(2*a), sp.sqrt(-disc)/(2*a)
    return sp.exp(lam*X)*sp.cos(mu*X), sp.exp(lam*X)*sp.sin(mu*X)


def sandbox(a, b, c, alpha, G, x_range=(-2, 2)):
    "Find y_p for a y'' + b y' + c y = e^(alpha x) G(x), check it, and plot the family."
    G = sp.sympify(str(G).replace('^', '**'), locals={'x': X})
    u, s, p_alpha, dp_alpha = particular(a, b, c, alpha, G)
    alpha_s = sp.nsimplify(alpha)
    yp = u*sp.exp(alpha_s*X)
    y1, y2 = homogeneous(a, b, c)
    f = G*sp.exp(alpha_s*X)
    check = sp.simplify(a*yp.diff(X, 2) + b*yp.diff(X) + c*yp - f)
    k = sp.degree(G, X) if G != 0 else 0
    lhs = lin_latex((a, b, c), ("y''", "y'", 'y'))

    display(Markdown('\n\n'.join([
        f'**Equation:** $${lhs} = {sp.latex(f)}$$',
        rf"$p(\alpha)={sp.latex(p_alpha)}$ and $p'(\alpha)={sp.latex(dp_alpha)}$. " + CASES[s],
        f'**Reduced equation:** $${reduced_latex(a, dp_alpha, p_alpha, G)}$$',
        f'**Guess:** $y_p = {guess_form(alpha, k, s)}$',
        f'**Particular solution:** $y_p = {sp.latex(yp)}$',
        rf'**Check:** $L[y_p] - f = {sp.latex(check)}$',
        rf'**General solution:** $y = y_p + c_1\,{sp.latex(y1)} + c_2\,{sp.latex(y2)}$',
    ])))

    xs = np.linspace(*x_range, 400)
    to_np = lambda e: np.broadcast_to(sp.lambdify(X, e, 'numpy')(xs), xs.shape).astype(float)
    yp_n, y1_n, y2_n = to_np(yp), to_np(y1), to_np(y2)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, (c1, c2) in enumerate([(1, 0), (-1, 0), (0, 1), (0, -1), (0.5, 0.5), (-0.5, -0.5)]):
        ax.plot(xs, yp_n + c1*y1_n + c2*y2_n, color='0.6', linewidth=1.2,
                label='other solutions:  y_p + c1 y1 + c2 y2' if i == 0 else None)
    ax.plot(xs, yp_n, color='C0', linewidth=3, label='y_p')
    lo, hi = np.min(yp_n), np.max(yp_n)
    pad = max(1.0, 0.6*(hi - lo))
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlim(x_range)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
    ax.set_title('the particular solution is one curve in a whole family')
    fig.tight_layout()
    plt.show()
    plt.close(fig)


print('Ready.')

## 1. Tune the guess until the leftover disappears

Take the equation

$$y''-3y'+2y=(4x+1)\,e^{3x}.$$

Read it as a sentence: *some combination of $y$ and its derivatives has to come out to exactly
$(4x+1)e^{3x}$.* The right side is a degree 1 polynomial times $e^{3x}$, so the natural guess is
the same shape with unknown numbers:

$$y_p=(A+Bx)\,e^{3x}.$$

Plug any guess into the left side and subtract the right side. What remains is the
**leftover**. The guess solves the equation exactly when the leftover is zero for every $x$.

The left panel below shows the leftover. Every term in it carries a factor of $e^{3x}$, so the
right panel divides that factor out, and what is left is just a line. Your job is to flatten that
line onto zero with the two sliders.

Before you touch the sliders, predict:

1. One slider changes only the *height* of the line on the right. The other changes the *tilt*,
   and maybe the height too. Which is which?
2. Can you find $B$ without knowing $A$? Can you find $A$ without knowing $B$?

In [ ]:
from ipywidgets import interact, FloatSlider

def explore_guess(A=0.0, B=0.0):
    leftover_plot(1, -3, 2, 3, u=Polynomial([A, B]), G=Polynomial([1, 4]),
                  guess=f'({A:g} + {B:g}x) e^(3x)', x_range=(-1.5, 0.75), lims=(40, 12))

interact(
    explore_guess,
    A=FloatSlider(value=0, min=-4, max=2, step=0.5, description='A', continuous_update=False),
    B=FloatSlider(value=0, min=0, max=4, step=0.5, description='B', continuous_update=False),
);

### What to notice

**In your notes,** fill these in:

- The leftover divided by $e^{3x}$ is the line $(\ \_\_\_\_\ )\,x+(\ \_\_\_\_\ )$, in terms of $A$
  and $B$.
- The guess that works is $A=$ ____ and $B=$ ____.

<details>
<summary>Check yourself after you have written something</summary>

The line is $(2B-4)x+(2A+3B-1)$. Only $B$ controls the tilt, so you can flatten the tilt first:
$B=2$. The height depends on both, and with $B=2$ it is $2A+5$, which is zero at $A=-2.5$. So

$$y_p=(-2.5+2x)\,e^{3x}.$$

That order is not a coincidence. Matching coefficients always works from the top power down: the
highest power pins down the highest coefficient, and then each lower power only has one new
unknown in it. "Matching coefficients" is just the algebra version of flattening the line.

The right panel is the whole trick of Trench 5.4: once you divide out $e^{3x}$, an exponential
problem becomes a polynomial problem.
</details>

## 2. The slider that does nothing

Keep the same left side, but change the forcing to plain $e^x$:

$$y''-3y'+2y=e^{x}.$$

The obvious guess is $y_p=Ae^{x}$. Before running, predict:

1. What value of $A$ makes the leftover zero?

Then run the cell and drag $A$ around with the guess set to `A e^x`. Then try the other two
guesses in the menu. Finally, switch the equation to the second one,
$y''-2y'+y=e^x$, and try all three guesses again.

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown

EQUATIONS = {"y'' - 3y' + 2y = e^x": (1, -3, 2),
             "y'' - 2y' + y = e^x": (1, -2, 1)}
GUESSES = {'A e^x': 0, 'A x e^x': 1, 'A x^2 e^x': 2}

def explore_silent(equation, guess, A=1.0):
    a, b, c = EQUATIONS[equation]
    m = GUESSES[guess]
    leftover_plot(a, b, c, 1, u=Polynomial([0]*m + [A]), G=Polynomial([1]),
                  guess=guess.replace('A', f'{A:g}'), x_range=(-2, 1.5), lims=(6, 6))

interact(
    explore_silent,
    equation=Dropdown(options=list(EQUATIONS), description='equation'),
    guess=Dropdown(options=list(GUESSES), description='guess'),
    A=FloatSlider(value=1, min=-2, max=2, step=0.25, description='A', continuous_update=False),
);

### What changed, and what stayed the same?

**In your notes,** fill these in from the cell:

- For $y''-3y'+2y=e^x$: with `A e^x` the leftover is ________ no matter what $A$ is. The guess
  that works is ________.
- With `A x^2 e^x` on that same equation, can you make the leftover zero?
- For $y''-2y'+y=e^x$: which guess works, and with what $A$?

<details>
<summary>Check yourself after you have written something</summary>

The characteristic polynomial of $y''-3y'+2y$ is $p(r)=r^2-3r+2=(r-1)(r-2)$, so $e^x$ already
solves the *homogeneous* equation. Plugging in $Ae^x$ gives zero on the left for every $A$, and the
leftover is stuck at $-e^x$. The slider is disconnected from the answer.

Multiplying by $x$ fixes it: $y_p=-xe^x$. Multiplying by $x^2$ overshoots. The leftover becomes a
tilted line, $2A-2Ax-1$, and no single $A$ flattens it.

For $y''-2y'+y$, $p(r)=(r-1)^2$ has a double root at $1$. Now both $e^x$ and $xe^x$ solve the
homogeneous equation, so both guesses are dead, and $y_p=\tfrac12x^2e^x$.
</details>

### Why this happens

Write the guess as $y=u\,e^{\alpha x}$ for some function $u$. Then
$y'=(u'+\alpha u)e^{\alpha x}$ and $y''=(u''+2\alpha u'+\alpha^2u)e^{\alpha x}$, and collecting terms,

$$ay''+by'+cy=e^{\alpha x}\Big[a\,u''+\underbrace{(2a\alpha+b)}_{p'(\alpha)}\,u'+\underbrace{(a\alpha^2+b\alpha+c)}_{p(\alpha)}\,u\Big].$$

So the equation $ay''+by'+cy=e^{\alpha x}G(x)$ is the same as the **reduced equation**

$$a\,u''+p'(\alpha)\,u'+p(\alpha)\,u=G(x).$$

That is exactly what the right-hand panels have been plotting. Now count degrees. Taking a
derivative lowers the degree of a polynomial by one. If $p(\alpha)\neq0$, the $u$ term is there
and $u$ can have the same degree as $G$. If $p(\alpha)=0$, the $u$ term vanishes, everything on
the left has been differentiated at least once, and $u$ needs one extra power of $x$ to make up for
it. If $p'(\alpha)=0$ too, only $u''$ is left, and $u$ needs two.

## 3. Read it off the graph of $p$

The reduced equation says that all you need to know about $\alpha$ is two numbers: $p(\alpha)$ and
$p'(\alpha)$. On the graph of $p$, those are the **height** of the graph at $r=\alpha$ and the
**slope** of the graph there.

The cell below draws $p(r)$ with a marker at $r=\alpha$, a dashed line for the height, and a short
tangent segment for the slope. Underneath, it writes out the reduced equation, the form of the
guess, and the actual particular solution for $G(x)=x^k$.

Before you run anything, predict:

1. For $p(r)=r^2-3r+2$, as you slide $\alpha$ from $-1$ to $3$, at which values does the guess
   change? What happens to the guess in between them?
2. For $p(r)=r^2-2r+1$, what is special about the graph at $\alpha=1$?
3. For $p(r)=r^2-2r+5$, is there any $\alpha$ that needs an extra $x$? At $\alpha=1$ the tangent
   is flat. Does that matter?

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

CHAR_POLYS = {'r^2 - 3r + 2   (roots 1 and 2)': (1, -3, 2),
              'r^2 - 2r + 1   (double root at 1)': (1, -2, 1),
              'r^2 - 2r + 5   (no real roots)': (1, -2, 5)}

def explore_alpha(p_choice, alpha=0.0, k=0):
    a, b, c = CHAR_POLYS[p_choice]
    p = lambda r: a*r**2 + b*r + c
    u, s, p_alpha, dp_alpha = particular(a, b, c, alpha, X**k)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    r = np.linspace(-1.5, 3.5, 400)
    ax.plot(r, p(r), color='0.2', linewidth=2)
    ax.axhline(0, color='0.5', linewidth=1)
    pa, dpa = float(p_alpha), float(dp_alpha)
    ax.plot([alpha, alpha], [0, pa], color='C1', linestyle='--', linewidth=2,
            label=f'height  p(α) = {pa:g}')
    t = np.linspace(alpha - 0.6, alpha + 0.6, 2)
    ax.plot(t, pa + dpa*(t - alpha), color='C0', linewidth=3,
            label=f"slope  p'(α) = {dpa:g}")
    ax.plot(alpha, pa, 'o', markersize=11, color='C3' if s else 'C2',
            markeredgecolor='black', zorder=5)
    ax.set_xlim(-1.5, 3.5)
    ax.set_ylim(-1, 6)
    ax.set_xlabel('r')
    ax.set_title(f'p(r) = {fmt_poly(Polynomial([c, b, a])).replace("x", "r")},   α = {alpha:g}'
                 f'   →   {["no extra x", "one extra x", "two extra x’s"][s]}')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper center', framealpha=0.9)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

    display(Markdown('\n\n'.join([
        CASES[s],
        f'**Reduced equation:** $${reduced_latex(a, dp_alpha, p_alpha, X**k)}$$',
        f'**Guess:** $y_p = {guess_form(alpha, k, s)}$',
        rf'**Particular solution:** $y_p = {sp.latex(u*sp.exp(sp.nsimplify(alpha)*X))}$',
    ])))

interact(
    explore_alpha,
    p_choice=Dropdown(options=list(CHAR_POLYS), description='p(r)'),
    alpha=FloatSlider(value=0, min=-1, max=3, step=0.25, description='α', continuous_update=False),
    k=IntSlider(value=0, min=0, max=2, description='deg G', continuous_update=False),
);

### What to notice

The marker turns red exactly when it lands on the $r$ axis, which is exactly when $\alpha$ is a
root of $p$. Away from the roots, nothing about the guess depends on $\alpha$ at all: the extra
$x$ shows up only *on* a root, not near one.

<details>
<summary>Check yourself after you have written something</summary>

(1) The guess changes only at $\alpha=1$ and $\alpha=2$, the two roots. At each one the height is
zero but the slope is not ($-1$ and $+1$), so the guess picks up one extra $x$. Everywhere else it
is the plain guess.

(2) At $\alpha=1$ the graph of $(r-1)^2$ touches the axis and turns around, so both the height
and the slope are zero. That is what a double root looks like, and it costs two extra $x$'s.

(3) No. The graph never reaches the axis, so $p(\alpha)$ is never zero and no extra $x$ is ever
needed. A flat tangent by itself does nothing: at $\alpha=1$ the reduced equation is
$u''+4u=G$, and the $u$ term is still there. Only the height decides whether you need an
extra $x$. The slope only matters once the height is already zero.

This polynomial has complex roots $1\pm2i$. In section 5.5 the forcing gets sines and cosines,
which means $\alpha$ is allowed to be complex too, and then it *can* land on those roots. That is
resonance.
</details>

## 4. Your turn: predict the guess, then check it

No scaffolding here. For each equation, work out on paper:

- $\alpha$, $p(\alpha)$, and $p'(\alpha)$,
- the form of the guess, with letters $A,B,\dots$ for the unknown numbers.

Then edit the numbers in the cell below and run it to check. It shows the reduced equation, the
guess, the particular solution, a check that the leftover really is zero, and a plot of $y_p$
alongside some other solutions of the same equation.

1. $y''+y'-6y=5e^{2x}$
2. $y''+4y'+4y=x\,e^{-2x}$
3. $y''-y=x^2e^{3x}$
4. $y''+2y'+5y=e^{-x}$

In the cell, the equation is $a\,y''+b\,y'+c\,y=e^{\alpha x}G(x)$. Type $G$ using `x`, with `*`
for multiplication and `^` for powers, like `3*x^2 - 1`. For no exponential, set `alpha = 0`.

In [ ]:
a, b, c = 1, -3, 2
alpha = 2
G = 'x'

sandbox(a, b, c, alpha, G)

<details>
<summary>Check yourself once you have written your answers down</summary>

(1) $p(r)=r^2+r-6=(r+3)(r-2)$ and $\alpha=2$: $p(2)=0$, $p'(2)=5$. One extra $x$:
$y_p=Axe^{2x}$, and the reduced equation $u''+5u'=5$ gives $y_p=xe^{2x}$.

(2) $p(r)=(r+2)^2$ and $\alpha=-2$: $p(-2)=0$ and $p'(-2)=0$. Two extra $x$'s on a degree 1
polynomial: $y_p=x^2(A+Bx)e^{-2x}$. The reduced equation is just $u''=x$, so
$y_p=\tfrac16x^3e^{-2x}$.

(3) $p(r)=r^2-1$ and $\alpha=3$: $p(3)=8$. No extra $x$, but $G$ has degree 2, so
$y_p=(A+Bx+Cx^2)e^{3x}$. Working down from the top power gives
$y_p=\big(\tfrac18x^2-\tfrac3{16}x+\tfrac7{64}\big)e^{3x}$.

(4) $p(r)=r^2+2r+5$ and $\alpha=-1$: $p(-1)=4$, $p'(-1)=0$. This is the trap. The slope is zero,
but the height is not, so no extra $x$: $y_p=Ae^{-x}$, and $4A=1$ gives $y_p=\tfrac14e^{-x}$.
</details>

## Wrap-up

You tuned a guess by hand, found a slider that could not do anything, and then predicted every
change to the guess from a picture of the characteristic polynomial.

The whole method is the reduced equation from the top. Given $ay''+by'+cy=e^{\alpha x}G(x)$, put
$y=u\,e^{\alpha x}$ and you get

$$a\,u''+p'(\alpha)\,u'+p(\alpha)\,u=G(x),$$

which is a polynomial problem. The rule about multiplying by $x$ or $x^2$ is not a separate fact
to memorize. It is just counting how many derivatives the lowest surviving term takes.

### Exit question

In two or three sentences, explain why the guess $y_p=Ae^{x}$ cannot work for
$y''-3y'+2y=e^{x}$ no matter what $A$ is, and why $y_p=Axe^x$ can. Use the phrase "homogeneous
equation" in your answer.